# CP5 – Ejercicios extra de PySpark


## Ejercicio 1 – Aeropuertos con mayor retraso medio

Usando el DataFrame `df_v`, agrupa los vuelos por `ORIGIN_AIRPORT` y calcula:

- El número total de vuelos.
- El retraso medio de llegada (`ARRIVAL_DELAY`).
- El retraso máximo de llegada.

Renombra las columnas resultantes con nombres claros y ordena el resultado de mayor a menor retraso medio.

Muestra únicamente los 10 primeros aeropuertos.

**Funciones a practicar:** `groupBy()`, `agg()`, `count()`, `avg()`, `max()`, `alias()`, `orderBy()`.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("CP5_extra").getOrCreate()
sc = spark.sparkContext
DATA_PATH = "/home/jovyan/data/"

df_v = spark.read.parquet(DATA_PATH + "vuelos")
df_e = spark.read.parquet(DATA_PATH + "estudiantes.parquet")
emple = spark.read.parquet(DATA_PATH + "empleados.parquet")
depart = spark.read.parquet(DATA_PATH + "departamentos.parquet")
df_calc = spark.read.parquet(DATA_PATH + "calculo.parquet")
df_c = spark.read.parquet(DATA_PATH + "colecciones")
df_p = spark.createDataFrame([('Jose', 1), ('Julia', 2), ('Katia', 1),
                              (None, 3), ('Raul', 3)], ['nombre', 'pago'])

In [4]:
(df_v.groupBy('ORIGIN_AIRPORT')
    .agg(
        F.count('*').alias('total_vuelos'),
        F.avg('ARRIVAL_DELAY').alias('retraso_medio_llegada'),
        F.max('ARRIVAL_DELAY').alias('retraso_max_llegada')
    )
    .orderBy(F.desc('retraso_medio_llegada'))
).show(10)


+--------------+------------+---------------------+-------------------+
|ORIGIN_AIRPORT|total_vuelos|retraso_medio_llegada|retraso_max_llegada|
+--------------+------------+---------------------+-------------------+
|         11905|           2|                390.0|                390|
|         14674|           2|                169.0|                354|
|         10333|           1|                 98.0|                 98|
|         12129|           4|                 95.0|                418|
|         14256|           2|                 92.5|                158|
|         13541|           1|                 43.0|                 43|
|           ALO|          21|    40.05263157894737|                401|
|           MEI|          28|   39.964285714285715|                702|
|         13256|           7|    38.57142857142857|                328|
|         15841|           2|                 35.0|                 81|
+--------------+------------+---------------------+-------------

## Ejercicio 2 – Distancia media por mes

Usando `df_v`, agrupa los vuelos por `MONTH` y calcula:

- La distancia media (`DISTANCE`).
- La distancia mínima.
- La distancia máxima.

Ordena el resultado por número de mes de forma ascendente.

**Funciones a practicar:** `groupBy()`, `agg()`, `avg()`, `min()`, `max()`.


In [5]:
(df_v.groupBy('MONTH')
    .agg(
        F.avg('DISTANCE').alias('distancia_media'),
        F.min('DISTANCE').alias('distancia_minima'),
        F.max('DISTANCE').alias('distancia_maxima')
    )
    .orderBy('MONTH')
).show(12)


+-----+-----------------+----------------+----------------+
|MONTH|  distancia_media|distancia_minima|distancia_maxima|
+-----+-----------------+----------------+----------------+
|    1|806.1322384255982|              67|            4983|
|    2|796.7817524424955|              31|            4962|
|    3|817.7122424878625|              31|            4983|
|    4|823.2305289462724|              31|            4962|
|    5|820.1246904490998|              31|            4983|
|    6|831.4370439788473|              31|            4962|
|    7| 834.798123563952|              31|            4983|
|    8|837.0570454396433|              31|            4983|
|    9|818.4249569954128|              31|            4983|
|   10|820.1984110816867|              31|            4983|
|   11|812.9152984285815|              67|            4983|
|   12|835.3684431389006|              31|            4983|
+-----+-----------------+----------------+----------------+



## Ejercicio 3 – Tabla pivotada de estudiantes

Usando `df_e`, crea una tabla dinámica en la que:

- Las filas representen `graduacion`.
- Las columnas correspondan a los valores de `sexo`.
- Los valores sean el promedio de `peso`.

Después, muestra el resultado.

**Funciones a practicar:** `groupBy()`, `pivot()`, `agg()`, `avg()`.


In [6]:
df_e.groupBy('graduacion').pivot('sexo').agg(
    F.avg('peso')
).show()


+----------+----+----+
|graduacion|   F|   M|
+----------+----+----+
|      2001|65.0|76.0|
|      2000|50.0|77.5|
+----------+----+----+



## Ejercicio 4 – Empleados con información de departamento

Combina los DataFrames `emple` y `depart` mediante un `inner join`.

La relación debe establecerse entre:

- `emple.num_dpto`
- `depart.id`

Muestra el resultado y comprueba qué información aporta cada DataFrame.

**Funciones a practicar:** `join()`, `col()`.


In [7]:
df_inner = emple.join(depart, emple['num_dpto'] == depart['id'], 'inner')
df_inner.show()


+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



## Ejercicio 5 – Departamentos sin empleados

Obtén únicamente los departamentos que no tienen ningún empleado asociado.

Parte de los DataFrames `depart` y `emple` y utiliza el tipo de join adecuado para obtener solo los registros de `depart` que no tienen correspondencia en `emple`.

**Funciones a practicar:** `join()` y `left anti join`.


In [8]:
depart.join(emple, depart['id'] == emple['num_dpto'], 'left_anti').show()


+---+-----------+
| id|nombre_dpto|
+---+-----------+
| 35|informática|
+---+-----------+



## Ejercicio 6 – Antigüedad de los registros

Usando `df_calc`, crea un nuevo DataFrame que muestre:

- `nombre`
- `fecha_ingreso`
- `fecha_salida`
- El número de días transcurridos entre ambas fechas.
- El número aproximado de meses transcurridos entre ambas fechas.

Renombra las dos columnas calculadas como `dias_transcurridos` y `meses_transcurridos`.

**Funciones a practicar:** `datediff()`, `months_between()`, `alias()`.


In [9]:
df_antig = df_calc.select(
    'nombre', 'fecha_ingreso', 'fecha_salida',
    F.datediff('fecha_salida', 'fecha_ingreso').alias('dias_transcurridos'),
    F.round(F.months_between('fecha_salida', 'fecha_ingreso'), 2).alias('meses_transcurridos')
)
df_antig.show()


+------+-------------+------------+------------------+-------------------+
|nombre|fecha_ingreso|fecha_salida|dias_transcurridos|meses_transcurridos|
+------+-------------+------------+------------------+-------------------+
|  Jose|   2021-01-01|  2021-11-14|               317|              10.42|
|Mayara|   2021-02-06|  2021-11-25|               292|               9.61|
+------+-------------+------------+------------------+-------------------+



## Ejercicio 7 – Descomponer una fecha y hora

Usando la columna `baja_sistema` de `df_calc`, crea una selección que incluya:

- Año.
- Mes.
- Día del mes.
- Día del año.
- Hora.
- Minuto.
- Segundo.

Asigna un alias descriptivo a cada columna.

**Funciones a practicar:** `year()`, `month()`, `dayofmonth()`, `dayofyear()`, `hour()`, `minute()`, `second()`.


In [14]:
df_calc.select(
    'baja_sistema',
    F.year('baja_sistema').alias('año'),
    F.month('baja_sistema').alias('mes'),
    F.dayofmonth('baja_sistema').alias('dia_del_mes'),
    F.dayofyear('baja_sistema').alias('dia_del_año'),
    F.hour('baja_sistema').alias('hora'),
    F.minute('baja_sistema').alias('minuto'),
    F.second('baja_sistema').alias('segundo')
).show(truncate=False)


+-------------------+----+---+-----------+-----------+----+------+-------+
|baja_sistema       |año |mes|dia_del_mes|dia_del_año|hora|minuto|segundo|
+-------------------+----+---+-----------+-----------+----+------+-------+
|2021-10-14 15:35:59|2021|10 |14         |287        |15  |35    |59     |
|2021-11-25 10:35:55|2021|11 |25         |329        |10  |35    |55     |
+-------------------+----+---+-----------+-----------+----+------+-------+



## Ejercicio 8 – Normalización de texto

Crea un DataFrame con una columna llamada `texto` que contenga varias cadenas con espacios al principio y al final.

A partir de esa columna, genera:

- El texto sin espacios exteriores.
- El texto en minúsculas.
- El texto en mayúsculas.
- El texto con la primera letra de cada palabra en mayúscula.
- El texto invertido.

**Funciones a practicar:** `trim()`, `lower()`, `upper()`, `initcap()`, `reverse()`.


In [11]:
df_t = spark.createDataFrame(
    [('  Hola Mundo  ',), ('  big DATA con spark ',), ('   PySpark  ',)],
    ['texto']
)

df_t.select(F.trim('texto').alias('sin_espacios')).select(
    'sin_espacios',
    F.lower('sin_espacios').alias('minuscula'),
    F.upper('sin_espacios').alias('mayuscula'),
    F.initcap('sin_espacios').alias('capital'),
    F.reverse('sin_espacios').alias('invertido')
).show(truncate=False)


+------------------+------------------+------------------+------------------+------------------+
|sin_espacios      |minuscula         |mayuscula         |capital           |invertido         |
+------------------+------------------+------------------+------------------+------------------+
|Hola Mundo        |hola mundo        |HOLA MUNDO        |Hola Mundo        |odnuM aloH        |
|big DATA con spark|big data con spark|BIG DATA CON SPARK|Big Data Con Spark|kraps noc ATAD gib|
|PySpark           |pyspark           |PYSPARK           |Pyspark           |krapSyP           |
+------------------+------------------+------------------+------------------+------------------+



## Ejercicio 9 – Analizar colecciones de tareas

Usando `df_c`, crea una transformación que muestre para cada `dia`:

- El número de elementos del array `tareas`.
- El array ordenado.
- Si el array contiene la tarea `buscar agua`.

Después, utiliza `explode()` para crear una segunda vista en la que cada tarea aparezca en una fila independiente.

**Funciones a practicar:** `size()`, `sort_array()`, `array_contains()`, `explode()`.


In [12]:
# Vista 1: tamaño, array ordenado y búsqueda
df_c.select(
    'dia',
    F.size('tareas').alias('num_tareas'),
    F.sort_array('tareas').alias('tareas_ordenadas'),
    F.array_contains('tareas', 'buscar agua').alias('contiene_buscar_agua')
).show(truncate=False)

# Vista 2: una tarea por fila
df_c.select(
    'dia',
    F.explode('tareas').alias('tarea')
).show(truncate=False)


+-----+----------+--------------------------------------------+--------------------+
|dia  |num_tareas|tareas_ordenadas                            |contiene_buscar_agua|
+-----+----------+--------------------------------------------+--------------------+
|lunes|3         |[buscar agua, hacer la tarea, lavar el auto]|true                |
+-----+----------+--------------------------------------------+--------------------+

+-----+--------------+
|dia  |tarea         |
+-----+--------------+
|lunes|hacer la tarea|
|lunes|buscar agua   |
|lunes|lavar el auto |
+-----+--------------+



## Ejercicio 10 – Clasificar y completar datos

Usando `df_p`, crea un nuevo DataFrame que:

1. Sustituya los valores nulos de `nombre` por el texto `sin nombre`.
2. Cree una columna `estado_pago` con estas reglas:
   - Si `pago == 1`: `pagado`.
   - Si `pago == 2`: `pendiente`.
   - Para cualquier otro valor: `sin iniciar`.
3. Muestre únicamente `nombre` y `estado_pago`.

**Funciones a practicar:** `coalesce()`, `lit()`, `when()`, `otherwise()`.


In [13]:
df_estado = df_p.select(
    F.coalesce(F.col('nombre'), F.lit('sin nombre')).alias('nombre'),
    (F.when(F.col('pago') == 1, 'pagado')
      .when(F.col('pago') == 2, 'pendiente')
      .otherwise('sin iniciar')).alias('estado_pago')
)
df_estado.show()


+----------+-----------+
|    nombre|estado_pago|
+----------+-----------+
|      Jose|     pagado|
|     Julia|  pendiente|
|     Katia|     pagado|
|sin nombre|sin iniciar|
|      Raul|sin iniciar|
+----------+-----------+

